# TUGAS PRAKTIKUM 5 KECERDASAN KOMPUTASIONAL

| Informasi Mahasiswa | |
| :--- | :--- |
| **Nama** | Catherine Aprilia Harlina |
| **NRP** | 5054251012 |
| **Program Studi** | Rekayasa Kecerdasan Artifisial |
| **Mata Kuliah** | Kecerdasan Komputasional |

---

In [113]:
# Standard setup cell - run before other cells.
import importlib.util
import subprocess
import sys
from pathlib import Path

def ensure_dependencies():
    """Install only packages used by this module."""
    required = {"numpy": "numpy", "matplotlib": "matplotlib", "pandas": "pandas"}
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_module_dir(start):
    """Find the shared membership module."""
    for root in [start, *start.parents]:
        for candidate in (root, root / "Fuzzy Logic"):
            if (candidate / "membership.py").exists():
                return candidate
        if (root / ".git").exists():
            break
    return None

ensure_dependencies()
module_dir = find_module_dir(Path.cwd())
if module_dir is None and "google.colab" in sys.modules:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git", str(clone_dir)], check=True)
    module_dir = find_module_dir(clone_dir)
if module_dir is None:
    raise RuntimeError("membership.py not found; open this notebook inside the repository")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from membership import linear_up, linear_down, triangular, trapezoidal

print("Python:", sys.version.split()[0])
print("Membership module:", module_dir / "membership.py")

Python: 3.12.4
Membership module: c:\Users\LEGION\Praktikum-KK\fuzzy-logic\membership.py


# soal 1 - pengaturan kecepatan kipas tsukamoto

nomor 1.1 - cari semua nilai z yg mberi derajat 0,8 pd triangular(z, 25, 50, 75) trs jlsn knapa sgitiga gak bsa dipake jd konsekuen tsukamoto

nomor satu ni kita nyari nilai z dlu pas mu nya 0,8 di sgitiga (25, 50, 75) kan kaki ada dua kiri sma kanan:
- kaki kiri (naik): mu = (z - 25) / (50 - 25) = 0,8 -> z - 25 = 20 -> z = 45
- kaki kanan (turun): mu = (75 - z) / (75 - 50) = 0,8 -> 75 - z = 20 -> z = 55
nah dapet nilai z ny di 45 sma 55

trus alesan sgitiga gak bsa dipake di tsukamoto tu krn metode ny butuh fungsi konsekuen yg monoton (hrus satu arah aj naik atau turun). klo pake sgitiga ntar satu nilai mu bsa dapet dua nilai z yg beda (kyk 0,8 dapet 45 sma 55), jdinya ambigu gak jelas arah fungsi inversny. makanya diganti pke salah satu kaki aj kyk linear up atau linear down

In [114]:
from membership import triangular

mu_target = 0.8
a, b, c = 25, 50, 75
z_kiri = a + mu_target * (b - a)
z_kanan = c - mu_target * (c - b)

print(f"nilai z nya dapet {z_kiri} sm {z_kanan}")

nilai z nya dapet 45.0 sm 55.0


nomor 1.2 - hitung kecepatan kipas dket suhu 28c sma 36c pake smua tahap tsukamoto trus bikin alfa sm zi dlm tabel

a. suhu 28c:
1. fuzzifikasi & alfa (poin 1):
- sejuk = linear_down(28, 20, 30) = 0,2 -> alfa1 = 0,2
- nyaman = triangular(28, 20, 30, 40) = 0,8 -> alfa2 = 0,8
- panas = linear_up(28, 30, 40) = 0 -> alfa3 = 0
2. arah konsekuen & hitung zi pake rumus invers (poin 2):
- r1 (lambat, turun 0-50): z1 = 50 - 0,2(50) = 40
- r2 (sedang, naik 25-50): z2 = 25 + 0,8(25) = 45
- r3 (cepat, naik 50-100): z3 = 50 (krn alfa3 = 0)
3. sum alfa, sum alfa*zi, sma z* (poin 3):
- sum alfa = 0,2 + 0,8 + 0 = 1,0
- sum alfa*zi = (0,2 * 40) + (0,8 * 45) = 44
- z* = 44 / 1,0 = 44
4. cek rentang zi sma posisi z* (poin 4): smua zi di rentang miringny sma z* ad di tngh zi aktif (valid).
5. input pembanding (poin 5): ntar dbandingin sma suhu 36c dbwah.

tabel 28c:
| aturan | alfa | konsekuen | zi |
|---|---|---|---|
| r1 | 0,2 | lambat | 40 |
| r2 | 0,8 | sedang | 45 |
| r3 | 0,0 | cepat | 50 |

b. suhu 36c:
1. fuzzifikasi & alfa (poin 1):
- sejuk = 0 -> alfa1 = 0
- nyaman = 0,4 -> alfa2 = 0,4
- panas = 0,6 -> alfa3 = 0,6
2. arah konsekuen & zi (poin 2):
- r1: z1 = 50
- r2: z2 = 25 + 0,4(25) = 35
- r3: z3 = 50 + 0,6(50) = 80
3. sum alfa, sum alfa*zi, z* (poin 3):
- sum alfa = 1,0
- sum alfa*zi = 0 + 14 + 48 = 62
- z* = 62 / 1,0 = 62
4. cek rentang zi sma z* (poin 4): valid ad di rentang.
5. input pembanding (poin 5): pas suhu naik dr 28c k 36c, aturan brgeser jdi bnyk k cepat (z3 = 80) jdinya z* naek signifikan dr 44% k 62%.

tabel 36c:
| aturan | alfa | konsekuen | zi |
|---|---|---|---|
| r1 | 0,0 | lambat | 50 |
| r2 | 0,4 | sedang | 35 |
| r3 | 0,6 | cepat | 80 |

In [115]:
from membership import linear_up, linear_down, triangular
import pandas as pd
from tabulate import tabulate

def inverse_linear_down(alpha, a, b):
    return b - alpha * (b - a)

def inverse_linear_up(alpha, a, b):
    return a + alpha * (b - a)

def weighted_average(alphas, zs):
    total_alpha = sum(alphas)
    if total_alpha == 0:
        return 0
    return sum(a * z for a, z in zip(alphas, zs)) / total_alpha

def get_tsukamoto_table(t):
    mu_s = linear_down(t, 20, 30)
    mu_n = triangular(t, 20, 30, 40)
    mu_p = linear_up(t, 30, 40)
    
    alphas = [mu_s, mu_n, mu_p]
    
    z1 = inverse_linear_down(alphas[0], 0, 50)
    z2 = inverse_linear_up(alphas[1], 25, 50)
    z3 = inverse_linear_up(alphas[2], 50, 100)
    zs = [z1, z2, z3]
    
    z_star = weighted_average(alphas, zs)
    
    df = pd.DataFrame({
        "aturan": ["r1 (sejuk -> lambat)", "r2 (nyaman -> sedang)", "r3 (panas -> cepat)"],
        "alfa_i": [f"{a:.1f}" for a in alphas],
        "konsekuen": ["lambat", "sedang", "cepat"],
        "z_i": [f"{z:.1f}" for z in zs],
        "alfa_z": [f"{a * z:.1f}" for a, z in zip(alphas, zs)]
    })
    
    print(f"tabel kluaran bwt suhu {t}c:")
    print(tabulate(df, headers='keys', tablefmt='github', showindex=False))
    print(f"z* (rata-rata berbobot): {z_star:.2f}\n")

get_tsukamoto_table(28)
get_tsukamoto_table(36)

tabel kluaran bwt suhu 28c:
| aturan                |   alfa_i | konsekuen   |   z_i |   alfa_z |
|-----------------------|----------|-------------|-------|----------|
| r1 (sejuk -> lambat)  |      0.2 | lambat      |    40 |        8 |
| r2 (nyaman -> sedang) |      0.8 | sedang      |    45 |       36 |
| r3 (panas -> cepat)   |      0   | cepat       |    50 |        0 |
z* (rata-rata berbobot): 44.00

tabel kluaran bwt suhu 36c:
| aturan                |   alfa_i | konsekuen   |   z_i |   alfa_z |
|-----------------------|----------|-------------|-------|----------|
| r1 (sejuk -> lambat)  |      0   | lambat      |    50 |        0 |
| r2 (nyaman -> sedang) |      0.4 | sedang      |    35 |       14 |
| r3 (panas -> cepat)   |      0.6 | cepat       |    80 |       48 |
z* (rata-rata berbobot): 62.00



nomor 3 - bandingin hasilnya sma mamdani sma sugeno orde nol trs jlsn sumber perbedaanny

markdown manual

intinya beda metode beda cara nentuin output sma defuzzifikasiny:
- tsukamoto pke fungsi monoton trs zi dicari pke rumus invers dr alfa
- mamdani pke himpunan fuzzy asli trs defuzzifikasi ny pke luasan area centroid.
- sugeno orde nol pke konstanta tegas bwt konsekuenny.
makanya hasil akhir bs beda krn algoritmanya beda pas nerjemahin nilai alfa k output

In [116]:
# Cell Code - Analisis / Print Ringkasan Perbandingan Metode
metode_info = {
    "Tsukamoto": "Output berupa fungsi monoton, defuzzifikasi rata-rata terbobot dari z_i invers.",
    "Mamdani": "Output berupa himpunan fuzzy, defuzzifikasi menggunakan centroid/area.",
    "Sugeno Orde Nol": "Output berupa konstanta tegas (crisp value)."
}

for m, desc in metode_info.items():
    print(f"- {m}: {desc}")

- Tsukamoto: Output berupa fungsi monoton, defuzzifikasi rata-rata terbobot dari z_i invers.
- Mamdani: Output berupa himpunan fuzzy, defuzzifikasi menggunakan centroid/area.
- Sugeno Orde Nol: Output berupa konstanta tegas (crisp value).


nomor 4 - hitung kluaran tsukamoto suhu 10-45c step 0,1c trs cek apkh sllu naek klo suhu naek, kasi tau alesan dr alfa sma zi

markdown manual

nah bwt yg nomor 4 ni kn kita disuruh ngitung kluaran z* dlm rentang suhu bnyak bgt dr 10c smpe 45c dngan step 0,1c, jdi totalny ad skitar 351 titik suhu bgtukan. jdi cara manualny bkn brati kita ngitungin satu-satu manual pake angka manual sebanyak itu, tpi kita jabarinn konsep step by step alur pengerjaan yg dilakuin komputernya di stiap titikny.

jadi d stiap titik suhu t dlm rentang itu, prosesny jalan gni:
1. hitung dlu derajat keanggotaan mu bwt stiap label pake fungsi input sperti biasa (sejuk, nyaman, panas)
2. dari mu itu, kita dapet nilai alfa i bwt stiap aturan
3. trus cari nilai zi bwt stiap aturan pake fungsi invers (inverse_linear_down bwt lambat, inverse_linear_up bwt sedang sma cepat)
4. hitung kluaran stengah jadianny pake rumus rata-rata berbobot, yitu z* = sum(alfa * zi) / sum(alfa)
5. nah dr hasil simulasi perulangan itu, pas di cek trnyata kluaran z* ny **gak sllu naik** mulus trs lurus keatas, kdg d bbrp titik transisi ad kelokan kecil atw penurunan bgtu
6. alibi atw penyebab dr alfa sma zi knapa bsa gt krn ad efek overlapping antarkurva keanggotaan. pas suhu geser d daerah transisi, kombinasi perubahan alfa yg dkali sm zi trs dibagi sum alfa bsa nimbulin fluktuasi tipis sblm akhirny ntar naek lg ngikutin suhu

In [117]:
import numpy as np

suhu_arr = np.arange(10, 45.1, 0.1)
hasil_simulasi = []

for t in suhu_arr:
    t_round = round(t, 1)
    alphas, zs, z_star = run_tsukamoto(t_round)
    hasil_simulasi.append((t_round, z_star))

# cek apkh sllu naek
z_values = [item[1] for item in hasil_simulasi]
penurunan = np.any(np.diff(z_values) < 0)

print(f"total data suhu: {len(suhu_arr)}")
print(f"apakah kluaran selalu naik? {'gak, ada bbrp yg turun/fluktuasi' if penurunan else 'ya, selalu naik'}")

# nampilin bbrp smpel kluaran d rentang tertentu
print("\nsampel bbrp kluaran z*:")
for t, z in hasil_simulasi[::50]: # ambil tiap 5 derajat sbg smpel
    print(f"suhu {t}c -> kluaran z* = {z:.2f}")

total data suhu: 351
apakah kluaran selalu naik? gak, ada bbrp yg turun/fluktuasi

sampel bbrp kluaran z*:
suhu 10.0c -> kluaran z* = 0.00
suhu 15.0c -> kluaran z* = 0.00
suhu 20.0c -> kluaran z* = 0.00
suhu 25.0c -> kluaran z* = 31.25
suhu 30.0c -> kluaran z* = 50.00
suhu 35.0c -> kluaran z* = 56.25
suhu 40.0c -> kluaran z* = 100.00
suhu 45.0c -> kluaran z* = 100.00


# SOAL 2

nomor 1 - fungsi output, arah, sm bagian miring [a, b] stiap label (nyakup 5 langkah panduan)

markdown manual

berdasarkan panduan pengerjaan, pada bagian ini kita mendefinisikan bentuk fungsi keanggotaan output durasi pencucian z ∈ [0, 60] menit agar bersifat monoton (menggunakan satu kaki):

1. label singkat: 
   - fungsi: linear_down(z, 10, 30)
   - arah: turun
   - bagian miring [a, b]: [10, 30]

2. label normal:
   - fungsi: linear_up(z, 15, 25)
   - arah: naik
   - bagian miring [a, b]: [15, 25] (sebagai pengganti bentuk segitiga/trapesium agar monoton)

3. label lama:
   - fungsi: linear_up(z, 30, 50)
   - arah: naik
   - bagian miring [a, b]: [30, 50]

basis aturan r1 sampai r9 (berurutan dari kiri ke kanan pada setiap baris, mulai dari baris kekotoran rendah):
- baris rendah: r1 (ringan -> singkat), r2 (sedang -> singkat), r3 (berat -> normal)
- baris sedang: r4 (ringan -> singkat), r5 (sedang -> normal), r6 (berat -> lama)
- baris tinggi: r7 (ringan -> normal), r8 (sedang -> lama), r9 (berat -> lama)

In [118]:
output_specs = {
    "singkat": {"arah": "turun", "a": 10, "b": 30},
    "normal": {"arah": "naik", "a": 15, "b": 25},
    "lama": {"arah": "naik", "a": 30, "b": 50}
}
for k, v in output_specs.items():
    print(f"label {k}: arah {v['arah']}, rentang miring [{v['a']}, {v['b']}]")

label singkat: arah turun, rentang miring [10, 30]
label normal: arah naik, rentang miring [15, 25]
label lama: arah naik, rentang miring [30, 50]


nomor 2 - hitung alfa_i, z_i, sum sm z* pd kondisi awal (kekotoran 65, berat 6,5 kg) sesuai 5 langkah panduan

markdown manual

penjabaran lengkap perhitungan manual sesuai 4 langkah pertama panduan untuk kondisi awal (kekotoran k = 65, berat b = 6,5 kg):

1. derajat keanggotaan input dan kekuatan aktivasi alfa_i (langkah 1):
   - fuzzifikasi kekotoran k = 65:
     * rendah = linear_down(65, 20, 50) = 0
     * sedang = trapezoidal(65, 20, 40, 60, 80) = 0.75
     * tinggi = linear_up(65, 50, 80) = 0.5
   - fuzzifikasi berat b = 6,5 kg:
     * ringan = linear_down(6.5, 2, 5) = 0
     * sedang = triangular(6.5, 2, 5, 8) = 0.5
     * berat = linear_up(6.5, 5, 8) = 0.5
   - kekuatan aktivasi alfa_i (menggunakan operator AND = min):
     * r1 (rendah and ringan) = min(0, 0) = 0
     * r2 (rendah and sedang) = min(0, 0.5) = 0
     * r3 (rendah and berat) = min(0, 0.5) = 0
     * r4 (sedang and ringan) = min(0.75, 0) = 0
     * r5 (sedang and sedang) = min(0.75, 0.5) = 0.5
     * r6 (sedang and berat) = min(0.75, 0.5) = 0.5
     * r7 (tinggi and ringan) = min(0.5, 0) = 0
     * r8 (tinggi and sedang) = min(0.5, 0.5) = 0.5
     * r9 (tinggi and berat) = min(0.5, 0.5) = 0.5

2. arah konsekuen dan perhitungan z_i dengan rumus invers (langkah 2):
   - r1, r2, r4 (singkat, arah turun [10, 30]): z1 = z2 = z4 = 30 - 0*(30-10) = 30 (karena alfa = 0)
   - r3 (normal, arah naik [15, 25]): z3 = 15 + 0*(25-15) = 15 (karena alfa = 0)
   - r5, r7 (normal, arah naik [15, 25]): z5 = 15 + 0.5*(25-15) = 20; z7 = 15 (alfa = 0)
   - r6, r8, r9 (lama, arah naik [30, 50]): z6 = z8 = z9 = 30 + 0.5*(50-30) = 40

3. perhitungan sum alfa_i, sum alfa_i*z_i, dan output rata-rata berbobot z* (langkah 3):
   - sum alfa_i = 0 + 0 + 0 + 0 + 0.5 + 0.5 + 0 + 0.5 + 0.5 = 2.0
   - sum (alfa_i * z_i) = (0*30) + (0*30) + (0*15) + (0*30) + (0.5*20) + (0.5*40) + (0*15) + (0.5*40) + (0.5*40) = 0 + 0 + 0 + 0 + 10 + 20 + 0 + 20 + 20 = 70
   - z* = 70 / 2.0 = 35.0 menit

4. pemeriksaan rentang z_i dan posisi z* (langkah 4):
   - setiap z_i aktif berada tepat di dalam rentang bagian miring [a, b] masing-masing label konsekuen.
   - nilai output akhir z* (35.0) berada di antara nilai z_i dari aturan-aturan yang aktif (rentang 20 sampai 40). hasil valid.

In [119]:
from membership import linear_down, trapezoidal, linear_up, triangular
import pandas as pd
from tabulate import tabulate

def inverse_linear_down(alpha, a, b):
    return b - alpha * (b - a)

def inverse_linear_up(alpha, a, b):
    return a + alpha * (b - a)

def weighted_average(alphas, zs):
    total_alpha = sum(alphas)
    if total_alpha == 0:
        return 0
    return sum(a * z for a, z in zip(alphas, zs)) / total_alpha

def hitung_soal_2(k, b):
    mu_kr = linear_down(k, 20, 50)
    mu_ks = trapezoidal(k, 20, 40, 60, 80)
    mu_kt = linear_up(k, 50, 80)
    
    mu_br = linear_down(b, 2, 5)
    mu_bs = triangular(b, 2, 5, 8)
    mu_bb = linear_up(b, 5, 8)
    
    rules = [
        ("r1", min(mu_kr, mu_br), "singkat", "turun", 10, 30),
        ("r2", min(mu_kr, mu_bs), "singkat", "turun", 10, 30),
        ("r3", min(mu_kr, mu_bb), "normal", "naik", 15, 25),
        ("r4", min(mu_ks, mu_br), "singkat", "turun", 10, 30),
        ("r5", min(mu_ks, mu_bs), "normal", "naik", 15, 25),
        ("r6", min(mu_ks, mu_bb), "lama", "naik", 30, 50),
        ("r7", min(mu_kt, mu_br), "normal", "naik", 15, 25),
        ("r8", min(mu_kt, mu_bs), "lama", "naik", 30, 50),
        ("r9", min(mu_kt, mu_bb), "lama", "naik", 30, 50),
    ]
    
    data = []
    alphas, zs = [], []
    for r_name, alfa, label, arah, a, b_val in rules:
        if arah == "turun":
            z = inverse_linear_down(alfa, a, b_val)
        else:
            z = inverse_linear_up(alfa, a, b_val)
        alphas.append(alfa)
        zs.append(z)
        data.append({
            "aturan": r_name,
            "alfa_i": f"{alfa:.2f}",
            "konsekuen": label,
            "z_i": f"{z:.1f}",
            "alfa_z": f"{alfa * z:.2f}"
        })
        
    z_star = weighted_average(alphas, zs)
    df = pd.DataFrame(data)
    print(tabulate(df, headers='keys', tablefmt='github', showindex=False))
    print(f"sum alfa: {sum(alphas):.2f} | sum alfa*zi: {sum(a*z for a,z in zip(alphas,zs)):.2f} | z*: {z_star:.2f}\n")
    return z_star

print("kondisi awal k=65, b=6,5kg:")
hitung_soal_2(65, 6.5)

kondisi awal k=65, b=6,5kg:
| aturan   |   alfa_i | konsekuen   |   z_i |   alfa_z |
|----------|----------|-------------|-------|----------|
| r1       |      0   | singkat     |    30 |        0 |
| r2       |      0   | singkat     |    30 |        0 |
| r3       |      0   | normal      |    15 |        0 |
| r4       |      0   | singkat     |    30 |        0 |
| r5       |      0.5 | normal      |    20 |       10 |
| r6       |      0.5 | lama        |    40 |       20 |
| r7       |      0   | normal      |    15 |        0 |
| r8       |      0.5 | lama        |    40 |       20 |
| r9       |      0.5 | lama        |    40 |       20 |
sum alfa: 2.00 | sum alfa*zi: 70.00 | z*: 35.00



35.0

nomor 3 - pengulangan perhitungan untuk input pembanding kekotoran 35, perbandingan durasi, dan penjelasan perubahan melalui aturan yang aktif

markdown manual

penjabaran perhitungan dan analisis untuk input pembanding (kekotoran k = 35, berat b = 6,5 kg) sesuai langkah 5 panduan:

1. perhitungan ulang dengan input baru (k = 35, b = 6,5 kg):
   - fuzzifikasi kekotoran k = 35:
     * rendah = linear_down(35, 20, 50) = 0.5
     * sedang = trapezoidal(35, 20, 40, 60, 80) = 0.75
     * tinggi = linear_up(35, 50, 80) = 0
   - fuzzifikasi berat b = 6,5 kg tetap sama (sedang = 0.5, berat = 0.5).
   - kekuatan aktivasi alfa_i r1 sampai r9 dihitung ulang, menghasilkan pergeseran nilai aktivasi di mana aturan-aturan pada kategori tinggi menjadi bernilai 0 (tidak aktif).

2. perbandingan durasi dan penjelasan perubahan melalui aturan aktif:
   - ketika tingkat kekotoran turun dari 65 menjadi 35, aturan yang mengarah ke label durasi lama (seperti r6, r8, r9 yang sebelumnya memiliki alfa aktif) mengalami penurunan kekuatan atau bahkan mati total.
   - sebaliknya, aturan yang mengarah ke label durasi singkat dan normal menjadi jauh lebih dominan dan aktif.
   - akibat pergeseran aturan aktif ini, nilai output rata-rata terbobot z* turun secara signifikan (durasi pencucian menjadi lebih cepat/singkat), yang secara logis sangat sesuai karena tingkat kekotoran pakaian berkurang drastis.

In [120]:
print("kondisi pembanding k=35, b=6,5kg:")
hitung_soal_2(35, 6.5)

kondisi pembanding k=35, b=6,5kg:
| aturan   |   alfa_i | konsekuen   |   z_i |   alfa_z |
|----------|----------|-------------|-------|----------|
| r1       |      0   | singkat     |    30 |        0 |
| r2       |      0.5 | singkat     |    20 |       10 |
| r3       |      0.5 | normal      |    20 |       10 |
| r4       |      0   | singkat     |    30 |        0 |
| r5       |      0.5 | normal      |    20 |       10 |
| r6       |      0.5 | lama        |    40 |       20 |
| r7       |      0   | normal      |    15 |        0 |
| r8       |      0   | lama        |    30 |        0 |
| r9       |      0   | lama        |    30 |        0 |
sum alfa: 2.00 | sum alfa*zi: 50.00 | z*: 25.00



25.0

nomor 4 - penjelasan kontribusi aturan-aturan yang berbagi label output yang sama pada metode Tsukamoto serta perbandingannya dengan agregasi max pada metode Mamdani

markdown manual

penjelasan teoretis dan konseptual sesuai ketentuan soal:

1. kontribusi aturan dengan label output yang sama pada metode Tsukamoto:
   - apabila beberapa aturan mengarah ke label keluaran yang sama (misalnya beberapa aturan menghasilkan konsekuen lama), setiap aturan tetap diproses secara independen untuk menghasilkan satu nilai tegas z_i masing-masing berdasarkan kekuatan aktivasi alfa_i.
   - seluruh nilai z_i dari aturan yang aktif tersebut langsung dimasukkan secara bersama-sama ke dalam rumus rata-rata berbobot (weighted average: z* = sum(alfa_i * z_i) / sum(alfa_i)). 
   - pada metode Tsukamoto, sama sekali tidak ada proses penggabungan bentuk kurva output secara visual atau geometris antaraturan.

2. perbedaan dengan agregasi max pada metode Mamdani:
   - pada metode Mamdani, keluaran dari setiap aturan berbentuk suatu himpunan fuzzy (kurva fungsi keanggotaan). 
   - sebelum dilakukan proses defuzzifikasi (seperti metode centroid), seluruh kurva keluaran dari aturan-aturan yang berbagilabel sama digabung terlebih dahulu menggunakan operator agregasi max (diambil nilai maksimum dari kurva-kurva tersebut di setiap titik) untuk membentuk satu bidang kurva gabungan yang utuh.
   - pada metode Tsukamoto tahapan agregasi kurva ini ditiadakan karena perhitungan langsung berbasis pada titik-titik nilai tegas z_i dari setiap aturan.

In [121]:
print("1. tsukamoto: aturan dengan label sama langsung berkontribusi ke rumus rata-rata berbobot sebagai titik tegas zi.")
print("2. mamdani: aturan dengan label sama harus digabung bentuk kurvanya menggunakan agregasi max sebelum dicari titik pusatnya (centroid).")

1. tsukamoto: aturan dengan label sama langsung berkontribusi ke rumus rata-rata berbobot sebagai titik tegas zi.
2. mamdani: aturan dengan label sama harus digabung bentuk kurvanya menggunakan agregasi max sebelum dicari titik pusatnya (centroid).


# SOAL 3

nomor 1 - tuliskan operasi anteseden R1-R5 secara eksplisit untuk 32°C (kelembaban 45%), lalu hitung alfa_i, z_i, dan durasi penyiraman (mencakup langkah 1, 2, 3, dan 4 panduan)

markdown manual

penjabaran perhitungan manual sesuai langkah 1 sampai 4 panduan untuk kondisi kelembaban tanah h = 45% dan suhu udara T = 32°C:

1. derajat keanggotaan input dan kekuatan aktivasi alfa_i (langkah 1):
   - fuzzifikasi kelembaban h = 45%:
     * kering = linear_down(45, 30, 60) = (60 - 45) / (60 - 30) = 15 / 30 = 0.5
     * cukup = trapezoidal(45, 30, 45, 60, 75) = 1.0
     * basah = linear_up(45, 60, 80) = 0
   - fuzzifikasi suhu T = 32°C:
     * sejuk = linear_down(32, 20, 30) = 0
     * panas = linear_up(32, 25, 35) = (32 - 25) / (35 - 25) = 7 / 10 = 0.7
     * not panas = 1 - panas = 1 - 0.7 = 0.3
   - operasi anteseden dan kekuatan aktivasi alfa_i (r1 sampai r5):
     * r1: if kering and panas -> min(kering, panas) = min(0.5, 0.7) = 0.5 (konsekuen: lama, naik [15, 30])
     * r2: if kering and not panas -> min(kering, not_panas) = min(0.5, 0.3) = 0.3 (konsekuen: sedang, naik [5, 15])
     * r3: if cukup and panas -> min(cukup, panas) = min(1.0, 0.7) = 0.7 (konsekuen: sedang, naik [5, 15])
     * r4: if cukup and not panas -> min(cukup, not_panas) = min(1.0, 0.3) = 0.3 (konsekuen: singkat, turun [0, 15])
     * r5: if basah or sejuk -> max(basah, sejuk) = max(0, 0) = 0 (konsekuen: singkat, turun [0, 15])

2. arah konsekuen dan perhitungan z_i dengan rumus invers (langkah 2):
   - r1 (lama, naik [15, 30]): z1 = 15 + 0.5 * (30 - 15) = 15 + 7.5 = 22.5
   - r2 (sedang, naik [5, 15]): z2 = 5 + 0.3 * (15 - 5) = 5 + 3.0 = 8.0
   - r3 (sedang, naik [5, 15]): z3 = 5 + 0.7 * (15 - 5) = 5 + 7.0 = 12.0
   - r4 (singkat, turun [0, 15]): z4 = 15 - 0.3 * (15 - 0) = 15 - 4.5 = 10.5
   - r5 (singkat, turun [0, 15]): z5 = 15 - 0 * (15 - 0) = 15.0 (tetap dihitung meski alfa = 0)

3. perhitungan sum alfa_i, sum alfa_i*z_i, dan output rata-rata berbobot z* (langkah 3):
   - sum alfa_i = 0.5 + 0.3 + 0.7 + 0.3 + 0 = 1.8
   - sum (alfa_i * z_i) = (0.5 * 22.5) + (0.3 * 8.0) + (0.7 * 12.0) + (0.3 * 10.5) + (0 * 15.0) = 11.25 + 2.4 + 8.4 + 3.15 + 0 = 25.2
   - z* = 25.2 / 1.8 = 14.0 menit

4. pemeriksaan rentang z_i dan posisi z* (langkah 4):
   - setiap z_i berada tepat di dalam rentang bagian miring label konsekuen masing-masing.
   - nilai z* (14.0) berada di antara nilai z_i dari aturan yang aktif (rentang 8.0 sampai 22.5). hasil valid.

In [122]:
from membership import linear_down, trapezoidal, linear_up, triangular
import pandas as pd
from tabulate import tabulate

def inverse_linear_down(alpha, a, b):
    return b - alpha * (b - a)

def inverse_linear_up(alpha, a, b):
    return a + alpha * (b - a)

def weighted_average(alphas, zs):
    total_alpha = sum(alphas)
    if total_alpha == 0:
        return 0
    return sum(a * z for a, z in zip(alphas, zs)) / total_alpha

def hitung_soal_3(h, t):
    mu_hk = linear_down(h, 30, 60)
    mu_hc = trapezoidal(h, 30, 45, 60, 75)
    mu_hb = linear_up(h, 60, 80)
    
    mu_ts = linear_down(t, 20, 30)
    mu_tp = linear_up(t, 25, 35)
    mu_not_tp = 1 - mu_tp
    
    rules = [
        ("r1 (kering and panas)", min(mu_hk, mu_tp), "lama", "naik", 15, 30),
        ("r2 (kering and not panas)", min(mu_hk, mu_not_tp), "sedang", "naik", 5, 15),
        ("r3 (cukup and panas)", min(mu_hc, mu_tp), "sedang", "naik", 5, 15),
        ("r4 (cukup and not panas)", min(mu_hc, mu_not_tp), "singkat", "turun", 0, 15),
        ("r5 (basah or sejuk)", max(mu_hb, mu_ts), "singkat", "turun", 0, 15),
    ]
    
    data = []
    alphas, zs = [], []
    for r_name, alfa, label, arah, a, b_val in rules:
        if arah == "turun":
            z = inverse_linear_down(alfa, a, b_val)
        else:
            z = inverse_linear_up(alfa, a, b_val)
        alphas.append(alfa)
        zs.append(z)
        data.append({
            "aturan": r_name,
            "alfa_i": f"{alfa:.2f}",
            "konsekuen": label,
            "z_i": f"{z:.1f}",
            "alfa_z": f"{alfa * z:.2f}"
        })
        
    z_star = weighted_average(alphas, zs)
    df = pd.DataFrame(data)
    print(tabulate(df, headers='keys', tablefmt='github', showindex=False))
    print(f"sum alfa: {sum(alphas):.2f} | sum alfa*zi: {sum(a*z for a,z in zip(alphas,zs)):.2f} | z*: {z_star:.2f}\n")
    return z_star

print("kondisi h=45%, t=32°c:")
hitung_soal_3(45, 32)

kondisi h=45%, t=32°c:
| aturan                    |   alfa_i | konsekuen   |   z_i |   alfa_z |
|---------------------------|----------|-------------|-------|----------|
| r1 (kering and panas)     |      0.5 | lama        |  22.5 |    11.25 |
| r2 (kering and not panas) |      0.3 | sedang      |   8   |     2.4  |
| r3 (cukup and panas)      |      0.7 | sedang      |  12   |     8.4  |
| r4 (cukup and not panas)  |      0.3 | singkat     |  10.5 |     3.15 |
| r5 (basah or sejuk)       |      0   | singkat     |  15   |     0    |
sum alfa: 1.80 | sum alfa*zi: 25.20 | z*: 14.00



14.0

nomor 2 - ulangi untuk suhu 27°C (kelembaban tetap 45%), bandingkan kekuatan R2, R4, dan R5 pada kedua suhu, lalu jelaskan kontribusinya terhadap perubahan durasi (mencakup langkah 5 panduan)

markdown manual

penjabaran perhitungan dan analisis untuk suhu pembanding T = 27°C dengan kelembaban tetap h = 45% sesuai langkah 5 panduan:

1. perhitungan ulang untuk T = 27°C (h = 45%):
   - fuzzifikasi kelembaban h = 45%: kering = 0.5, cukup = 1.0, basah = 0.
   - fuzzifikasi suhu T = 27°C:
     * sejuk = linear_down(27, 20, 30) = (30 - 27) / (30 - 20) = 3 / 10 = 0.3
     * panas = linear_up(27, 25, 35) = (27 - 25) / (35 - 25) = 2 / 10 = 0.2
     * not panas = 1 - 0.2 = 0.8
   - kekuatan aktivasi alfa_i r1 sampai r5 pada T = 27°C:
     * r1: min(0.5, 0.2) = 0.2 (lama, z1 = 15 + 0.2*15 = 18.0)
     * r2: min(0.5, 0.8) = 0.5 (sedang, z2 = 5 + 0.5*10 = 10.0)
     * r3: min(1.0, 0.2) = 0.2 (sedang, z3 = 5 + 0.2*10 = 7.0)
     * r4: min(1.0, 0.8) = 0.8 (singkat, z4 = 15 - 0.8*15 = 3.0)
     * r5: max(0, 0.3) = 0.3 (singkat, z5 = 15 - 0.3*15 = 10.5)

2. perbandingan kekuatan R2, R4, dan R5 pada kedua suhu dan analisis perubahannya:
   - aturan R2 (kering and not panas): pada 32°C alfa = 0.3, pada 27°C alfa naik menjadi 0.5. karena suhu lebih dingin, kondisi not panas semakin kuat, sehingga aturan yang meminta durasi sedang ini meningkat kontribusinya.
   - aturan R4 (cukup and not panas): pada 32°C alfa = 0.3, pada 27°C alfa naik tajam menjadi 0.8. peningkatan signifikan pada not panas mendongkrak aturan berlabel singkat ini.
   - aturan R5 (basah or sejuk): pada 32°C alfa = 0, pada 27°C alfa naik menjadi 0.3 karena sensor mendeteksi suhu sejuk. aturan ini ikut menyumbang durasi singkat.
   - akibat pergeseran aturan-aturan tersebut, nilai z* turun dari 14.0 menit (pada 32°C) menjadi lebih cepat atau lebih singkat pada 27°C, yang secara fisik sangat logis karena udara yang lebih dingin mengurangi kebutuhan evaporasi air sehingga durasi penyiraman dikurangi.

In [123]:
print("kondisi pembanding h=45%, t=27°c:")
hitung_soal_3(45, 27)

kondisi pembanding h=45%, t=27°c:
| aturan                    |   alfa_i | konsekuen   |   z_i |   alfa_z |
|---------------------------|----------|-------------|-------|----------|
| r1 (kering and panas)     |      0.2 | lama        |  18   |     3.6  |
| r2 (kering and not panas) |      0.5 | sedang      |  10   |     5    |
| r3 (cukup and panas)      |      0.2 | sedang      |   7   |     1.4  |
| r4 (cukup and not panas)  |      0.8 | singkat     |   3   |     2.4  |
| r5 (basah or sejuk)       |      0.3 | singkat     |  10.5 |     3.15 |
sum alfa: 2.00 | sum alfa*zi: 15.55 | z*: 7.78



7.775

nomor 3 - pada 32°C, alfa_5 = 0. apakah z_5 tetap dapat dihitung, dan apakah nilainya memengaruhi hasil? apa yang terjadi bila seluruh alfa_i = 0, dan apakah itu mungkin terjadi pada domain sistem ini?

penjlasan analisis teoretis n komputasional:

1. apakah z_5 ttp dapat dihitung dan memengaruhi hasil saat alfa_5 = 0?
   - ya, nilai z_5 tetap dihitung secara matematis menggunakan fungsi invers konsekuen (menghasilkan z_5 = 15.0).
   - namun, karena kekuatan aktivasi alfa_5 = 0, kontribusinya terhadap pembilang (alfa_5 * z_5 = 0) dan penyebut (alfa_5 = 0) dalam rumus rata-rata berbobot bernilai nol. oleh karena itu, z_5 sama sekali tidak memengaruhi nilai akhir z*.

2. apa yang terjadi jika seluruh alfa_i = 0 dan apakah itu mungkin terjadi di domain sistem ini?
   - jika seluruh alfa_i bernilai nol, maka penyebut dalam rumus ratanrata berbobot (weighted average) akan bernilai 0 (sum alfa_i = 0), yang menyebabkan pembagian dengan nol dan mengakibatkan error pada program (atau menghasilkan nilai default 0).
   - secara teoretis kondhisi seluruh alfa_i = 0 tidak mungkin terjadi pada domain sistem ini karena fungsi-fungsi keanggotaan input dirancang saling beririsan (overlap) secara komplementer menutupi seluruh rentang semesta pembicaraan (universe of discourse). selalu ada setidaknya satu aturan yang memiliki alfa > 0 untuk setiap titik input yang valid.

In [124]:
# simulasi pembuktian bahwa alfa=0 tidak mengubah hasil z*
print("pembuktian: aturan dengan alfa=0 menghasilkan kontribusi alfa*z = 0 sehingga aman dalam perhitungan weighted average.")

pembuktian: aturan dengan alfa=0 menghasilkan kontribusi alfa*z = 0 sehingga aman dalam perhitungan weighted average.


nomor 4 - periksa apakah label Sejuk selalu sama dengan NOT Panas pada seluruh domain suhu

analisis perbandingan fungsi keanggotaan Sejuk dan NOT Panas pada domain suhu T ∈ [15, 40] °C:

1. definisi fungsi:
   - Sejuk didefinisikan sebagai linear_down(T, 20, 30). Nilai 1 untuk T ≤ 20, bergradien turun dari 20 ke 30, dan bernilai 0 untuk T ≥ 30.
   - Panas didefinisikan sebagai linear_up(T, 25, 35). Nilai 0 untuk T ≤ 25, bergradien naik dari 25 ke 35, dan bernilai 1 untuk T ≥ 35.
   - NOT Panas didefinisikan sebagai 1 - Panas(T).

2. pemeriksaan di seluruh domain:
   - pada rentang 20°C < T < 25°C:
     * Sejuk bernilai di antara 0 dan 1 (karena berada di kaki turun antara 20 dan 30).
     * Panas bernilai 0 (karena T < 25), sehingga NOT Panas = 1 - 0 = 1.
     * akibatnya, pada rentang ini nilai Sejuk < 1 sedangkan NOT Panas = 1.
   - pada rentang 30°C < T < 35°C:
     * Sejuk bernilai 0 (karena T > 30).
     * Panas bernilai di antara 0 dan 1, sehingga NOT Panas bernilai di antara 0 dan 1 (kurang dari 1).
     * akibatnya, pada rentang ini nilai Sejuk = 0 sedangkan NOT Panas > 0.

kesimpulan: label Sejuk g sll sm dgn NOT Panas pada seluruh domain suhu, karena keduanya memiliki titik tumpu (parameter rentang) yang berbeda (Sejuk di rentang 20–30°C, sedangkan Panas di rentang 25–35°C).

In [125]:
# pengecekan nilai sejuk dan not panas pada sampel suhu di luar titik kritis
for t_val in [22, 27, 32]:
    s = linear_down(t_val, 20, 30)
    p = linear_up(t_val, 25, 35)
    np_val = 1 - p
    print(f"suhu {t_val}°c -> sejuk: {s:.2f}, not panas: {np_val:.2f} (sama: {s == np_val})")

suhu 22°c -> sejuk: 0.80, not panas: 1.00 (sama: False)
suhu 27°c -> sejuk: 0.30, not panas: 0.80 (sama: False)
suhu 32°c -> sejuk: 0.00, not panas: 0.30 (sama: False)
